In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import boto3
import joblib
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, auc
from sklearn.metrics import precision_recall_curve, roc_curve
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

## Helper Classes

In [ ]:
class ModelComparison:
    def __init__(self, str_bucket, str_dirname_output):
        self.str_bucket = str_bucket
        self.str_dirname_output = str_dirname_output
        self.s3_client = boto3.client('s3')
        self.df_valid = None
        self.df_test = None
        self.arr_if_scores_valid = None
        self.arr_lof_scores_valid = None
        self.arr_if_scores = None
        self.arr_lof_scores = None
        self.dict_if_metrics = {}
        self.dict_lof_metrics = {}
        self.flt_if_threshold = None
        self.flt_lof_threshold = None

    def import_data_and_scores(self):
        """Load valid/test sets and model predictions from S3."""
        print("Loading data and model scores...")
        
        str_valid_uri = f's3://{self.str_bucket}/02_preprocessing/valid_processed.parquet'
        str_test_uri = f's3://{self.str_bucket}/02_preprocessing/test_processed.parquet'
        self.df_valid = pd.read_parquet(str_valid_uri)
        self.df_test = pd.read_parquet(str_test_uri)
        
        # Load models from S3
        from io import BytesIO
        
        obj_if = self.s3_client.get_object(Bucket=self.str_bucket, Key='03_isolation_forest/isolation_forest_model.pkl')
        self.model_if = joblib.load(BytesIO(obj_if['Body'].read()))
        
        obj_lof = self.s3_client.get_object(Bucket=self.str_bucket, Key='04_lof/lof_model.pkl')
        self.model_lof = joblib.load(BytesIO(obj_lof['Body'].read()))
        
        list_feature_cols = [col for col in self.df_test.columns if col != 'isFraud']
        
        # Score validation and test sets
        self.arr_if_scores_valid = self.model_if.score_samples(self.df_valid[list_feature_cols])
        self.arr_lof_scores_valid = self.model_lof.score_samples(self.df_valid[list_feature_cols])
        self.arr_if_scores = self.model_if.score_samples(self.df_test[list_feature_cols])
        self.arr_lof_scores = self.model_lof.score_samples(self.df_test[list_feature_cols])
        
        print(f"Validation set: {self.df_valid.shape}")
        print(f"Test set: {self.df_test.shape}")
        print(f"IF test scores: mean={self.arr_if_scores.mean():.4f}, std={self.arr_if_scores.std():.4f}")
        print(f"LOF test scores: mean={self.arr_lof_scores.mean():.4f}, std={self.arr_lof_scores.std():.4f}")

    def compute_metrics_both_models(self):
        """Compute evaluation metrics for both models (threshold from validation, metrics on test)."""
        if self.arr_if_scores is None or self.arr_lof_scores is None:
            raise ValueError("Scores not loaded.")
        
        print("\nComputing metrics for both models...")
        print("(Thresholds selected on validation set, metrics reported on test set)")
        
        arr_y_valid = self.df_valid['isFraud'].values
        arr_y_test = self.df_test['isFraud'].values
        
        # --- Isolation Forest ---
        # Find threshold on validation set
        arr_if_valid_neg = -self.arr_if_scores_valid
        arr_if_prec_v, arr_if_rec_v, arr_if_thresh_v = precision_recall_curve(arr_y_valid, arr_if_valid_neg)
        list_if_f1_v = []
        for flt_threshold in arr_if_thresh_v:
            arr_preds = (arr_if_valid_neg >= flt_threshold).astype(int)
            list_if_f1_v.append(f1_score(arr_y_valid, arr_preds) if arr_preds.sum() > 0 else 0)
        int_if_best_idx = np.argmax(list_if_f1_v)
        self.flt_if_threshold = arr_if_thresh_v[int_if_best_idx] if int_if_best_idx < len(arr_if_thresh_v) else arr_if_thresh_v[-1]
        
        # Evaluate on test set
        arr_if_scores_neg = -self.arr_if_scores
        flt_if_roc_auc = roc_auc_score(arr_y_test, arr_if_scores_neg)
        arr_if_precision, arr_if_recall, _ = precision_recall_curve(arr_y_test, arr_if_scores_neg)
        flt_if_pr_auc = auc(arr_if_recall, arr_if_precision)
        arr_if_preds = (arr_if_scores_neg >= self.flt_if_threshold).astype(int)
        
        self.dict_if_metrics = {
            'Model': 'Isolation Forest',
            'ROC-AUC': flt_if_roc_auc,
            'PR-AUC': flt_if_pr_auc,
            'Precision': precision_score(arr_y_test, arr_if_preds),
            'Recall': recall_score(arr_y_test, arr_if_preds),
            'F1-Score': f1_score(arr_y_test, arr_if_preds),
            'Threshold': self.flt_if_threshold
        }
        
        # --- LOF ---
        # Find threshold on validation set
        arr_lof_valid_neg = -self.arr_lof_scores_valid
        arr_lof_prec_v, arr_lof_rec_v, arr_lof_thresh_v = precision_recall_curve(arr_y_valid, arr_lof_valid_neg)
        list_lof_f1_v = []
        for flt_threshold in arr_lof_thresh_v:
            arr_preds = (arr_lof_valid_neg >= flt_threshold).astype(int)
            list_lof_f1_v.append(f1_score(arr_y_valid, arr_preds) if arr_preds.sum() > 0 else 0)
        int_lof_best_idx = np.argmax(list_lof_f1_v)
        self.flt_lof_threshold = arr_lof_thresh_v[int_lof_best_idx] if int_lof_best_idx < len(arr_lof_thresh_v) else arr_lof_thresh_v[-1]
        
        # Evaluate on test set
        arr_lof_scores_neg = -self.arr_lof_scores
        flt_lof_roc_auc = roc_auc_score(arr_y_test, arr_lof_scores_neg)
        arr_lof_precision, arr_lof_recall, _ = precision_recall_curve(arr_y_test, arr_lof_scores_neg)
        flt_lof_pr_auc = auc(arr_lof_recall, arr_lof_precision)
        arr_lof_preds = (arr_lof_scores_neg >= self.flt_lof_threshold).astype(int)
        
        self.dict_lof_metrics = {
            'Model': 'Local Outlier Factor',
            'ROC-AUC': flt_lof_roc_auc,
            'PR-AUC': flt_lof_pr_auc,
            'Precision': precision_score(arr_y_test, arr_lof_preds),
            'Recall': recall_score(arr_y_test, arr_lof_preds),
            'F1-Score': f1_score(arr_y_test, arr_lof_preds),
            'Threshold': self.flt_lof_threshold
        }
        
        print("\nIsolation Forest (test set):")
        for str_key, val in self.dict_if_metrics.items():
            if str_key != 'Model':
                print(f"  {str_key}: {val:.4f}")
        
        print("\nLocal Outlier Factor (test set):")
        for str_key, val in self.dict_lof_metrics.items():
            if str_key != 'Model':
                print(f"  {str_key}: {val:.4f}")

    def plot_roc_comparison(self):
        """Plot ROC curves side by side."""
        if self.arr_if_scores is None or self.arr_lof_scores is None:
            raise ValueError("Scores not loaded.")
        
        arr_y_test = self.df_test['isFraud'].values
        arr_if_scores_neg = -self.arr_if_scores
        arr_lof_scores_neg = -self.arr_lof_scores
        
        flt_if_roc_auc = roc_auc_score(arr_y_test, arr_if_scores_neg)
        flt_lof_roc_auc = roc_auc_score(arr_y_test, arr_lof_scores_neg)
        
        arr_if_fpr, arr_if_tpr, _ = roc_curve(arr_y_test, arr_if_scores_neg)
        arr_lof_fpr, arr_lof_tpr, _ = roc_curve(arr_y_test, arr_lof_scores_neg)
        
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(arr_if_fpr, arr_if_tpr, linewidth=2.5, label=f'Isolation Forest (AUC={flt_if_roc_auc:.4f})', color='steelblue')
        ax.plot(arr_lof_fpr, arr_lof_tpr, linewidth=2.5, label=f'LOF (AUC={flt_lof_roc_auc:.4f})', color='coral')
        ax.plot([0, 1], [0, 1], linestyle='--', color='gray', linewidth=1.5, label='Random Classifier')
        
        ax.set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
        ax.set_ylabel('True Positive Rate', fontsize=12, fontweight='bold')
        ax.set_title('ROC Curve Comparison', fontsize=14, fontweight='bold')
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/01_roc_comparison.png', bbox_inches='tight', dpi=150)
        plt.close()
        print("Saved: 01_roc_comparison.png")

    def plot_pr_comparison(self):
        """Plot Precision-Recall curves side by side."""
        if self.arr_if_scores is None or self.arr_lof_scores is None:
            raise ValueError("Scores not loaded.")
        
        arr_y_test = self.df_test['isFraud'].values
        arr_if_scores_neg = -self.arr_if_scores
        arr_lof_scores_neg = -self.arr_lof_scores
        
        arr_if_precision, arr_if_recall, _ = precision_recall_curve(arr_y_test, arr_if_scores_neg)
        flt_if_pr_auc = auc(arr_if_recall, arr_if_precision)
        
        arr_lof_precision, arr_lof_recall, _ = precision_recall_curve(arr_y_test, arr_lof_scores_neg)
        flt_lof_pr_auc = auc(arr_lof_recall, arr_lof_precision)
        
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot(arr_if_recall, arr_if_precision, linewidth=2.5, label=f'Isolation Forest (AUC={flt_if_pr_auc:.4f})', color='steelblue')
        ax.plot(arr_lof_recall, arr_lof_precision, linewidth=2.5, label=f'LOF (AUC={flt_lof_pr_auc:.4f})', color='coral')
        
        ax.set_xlabel('Recall', fontsize=12, fontweight='bold')
        ax.set_ylabel('Precision', fontsize=12, fontweight='bold')
        ax.set_title('Precision-Recall Curve Comparison', fontsize=14, fontweight='bold')
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/02_pr_comparison.png', bbox_inches='tight', dpi=150)
        plt.close()
        print("Saved: 02_pr_comparison.png")

    def plot_metrics_comparison(self):
        """Plot metrics comparison as bar chart."""
        if not self.dict_if_metrics or not self.dict_lof_metrics:
            raise ValueError("Metrics not computed.")
        
        list_metrics = ['ROC-AUC', 'PR-AUC', 'Precision', 'Recall', 'F1-Score']
        list_if_values = [self.dict_if_metrics[m] for m in list_metrics]
        list_lof_values = [self.dict_lof_metrics[m] for m in list_metrics]
        
        fig, ax = plt.subplots(figsize=(12, 6))
        
        x = np.arange(len(list_metrics))
        width = 0.35
        
        ax.bar(x - width/2, list_if_values, width, label='Isolation Forest', color='steelblue', alpha=0.8, edgecolor='black')
        ax.bar(x + width/2, list_lof_values, width, label='LOF', color='coral', alpha=0.8, edgecolor='black')
        
        ax.set_ylabel('Score', fontsize=12, fontweight='bold')
        ax.set_title('Model Metrics Comparison', fontsize=14, fontweight='bold')
        ax.set_xticks(x)
        ax.set_xticklabels(list_metrics)
        ax.legend()
        ax.grid(True, alpha=0.3, axis='y')
        ax.set_ylim([0, 1])
        
        # Add value labels on bars
        for i, v in enumerate(list_if_values):
            ax.text(i - width/2, v + 0.02, f'{v:.3f}', ha='center', va='bottom', fontsize=9)
        for i, v in enumerate(list_lof_values):
            ax.text(i + width/2, v + 0.02, f'{v:.3f}', ha='center', va='bottom', fontsize=9)
        
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/03_metrics_comparison.png', bbox_inches='tight', dpi=150)
        plt.close()
        print("Saved: 03_metrics_comparison.png")

    def plot_score_distributions(self):
        """Plot anomaly score distributions for both models."""
        if self.arr_if_scores is None or self.arr_lof_scores is None:
            raise ValueError("Scores not loaded.")
        
        arr_y_test = self.df_test['isFraud'].values
        
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Isolation Forest
        axes[0].hist(-self.arr_if_scores[arr_y_test == 0], bins=100, alpha=0.6, label='Non-Fraud', color='#2ecc71')
        axes[0].hist(-self.arr_if_scores[arr_y_test == 1], bins=100, alpha=0.6, label='Fraud', color='#e74c3c')
        axes[0].axvline(self.flt_if_threshold, color='black', linestyle='--', linewidth=2)
        axes[0].set_xlabel('Anomaly Score', fontsize=11, fontweight='bold')
        axes[0].set_ylabel('Frequency', fontsize=11, fontweight='bold')
        axes[0].set_title('Isolation Forest Score Distribution', fontsize=12, fontweight='bold')
        axes[0].legend()
        axes[0].set_yscale('log')
        
        # LOF
        axes[1].hist(-self.arr_lof_scores[arr_y_test == 0], bins=100, alpha=0.6, label='Non-Fraud', color='#2ecc71')
        axes[1].hist(-self.arr_lof_scores[arr_y_test == 1], bins=100, alpha=0.6, label='Fraud', color='#e74c3c')
        axes[1].axvline(self.flt_lof_threshold, color='black', linestyle='--', linewidth=2)
        axes[1].set_xlabel('Anomaly Score', fontsize=11, fontweight='bold')
        axes[1].set_ylabel('Frequency', fontsize=11, fontweight='bold')
        axes[1].set_title('LOF Score Distribution', fontsize=12, fontweight='bold')
        axes[1].legend()
        axes[1].set_yscale('log')
        
        plt.tight_layout()
        plt.savefig(f'{self.str_dirname_output}/04_score_distributions.png', bbox_inches='tight', dpi=150)
        plt.close()
        print("Saved: 04_score_distributions.png")

    def ensemble_analysis(self):
        """Analyze ensemble predictions (both models agree)."""
        if self.arr_if_scores is None or self.arr_lof_scores is None:
            raise ValueError("Scores not loaded.")
        
        print("\nEnsemble Analysis (both models must agree):")
        
        arr_y_test = self.df_test['isFraud'].values
        arr_if_scores_neg = -self.arr_if_scores
        arr_lof_scores_neg = -self.arr_lof_scores
        
        arr_if_preds = (arr_if_scores_neg >= self.flt_if_threshold).astype(int)
        arr_lof_preds = (arr_lof_scores_neg >= self.flt_lof_threshold).astype(int)
        arr_ensemble_preds = (arr_if_preds & arr_lof_preds).astype(int)
        
        flt_ensemble_precision = precision_score(arr_y_test, arr_ensemble_preds) if arr_ensemble_preds.sum() > 0 else 0
        flt_ensemble_recall = recall_score(arr_y_test, arr_ensemble_preds) if arr_ensemble_preds.sum() > 0 else 0
        flt_ensemble_f1 = f1_score(arr_y_test, arr_ensemble_preds) if arr_ensemble_preds.sum() > 0 else 0
        
        print(f"  Flagged as anomalies: {arr_ensemble_preds.sum():,} ({arr_ensemble_preds.sum()/len(arr_y_test)*100:.2f}%)")
        print(f"  True positives: {(arr_ensemble_preds & arr_y_test).sum()}")
        print(f"  Precision: {flt_ensemble_precision:.4f}")
        print(f"  Recall: {flt_ensemble_recall:.4f}")
        print(f"  F1-Score: {flt_ensemble_f1:.4f}")
        
        return {
            'precision': flt_ensemble_precision,
            'recall': flt_ensemble_recall,
            'f1': flt_ensemble_f1,
            'flagged': arr_ensemble_preds.sum()
        }

    def save_comparison_table(self):
        """Save metrics comparison table to CSV."""
        df_comparison = pd.DataFrame([self.dict_if_metrics, self.dict_lof_metrics])
        str_path = os.path.join(self.str_dirname_output, 'metrics_comparison.csv')
        df_comparison.to_csv(str_path, index=False)
        print(f"\nMetrics comparison saved to: {str_path}")
        print(df_comparison.to_string(index=False))

## Constants

In [ ]:
str_bucket = 'anomaly-detection-demo-repo'
str_task = 'comparison'
str_dirname_output = './output'

## Output Directory

In [ ]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass
print(f"Output directory ready: {str_dirname_output}")

## Load Data and Model Scores

In [ ]:
comparison = ModelComparison(str_bucket, str_dirname_output)
comparison.import_data_and_scores()

## Compute Metrics

In [ ]:
comparison.compute_metrics_both_models()

## Generate Comparison Visualizations

In [ ]:
comparison.plot_roc_comparison()

In [ ]:
comparison.plot_pr_comparison()

In [ ]:
comparison.plot_metrics_comparison()

In [ ]:
comparison.plot_score_distributions()

## Ensemble Analysis

In [ ]:
dict_ensemble_metrics = comparison.ensemble_analysis()

## Save Results

In [ ]:
comparison.save_comparison_table()

## Summary

In [ ]:
print("\n=== MODEL COMPARISON COMPLETE ===")
print("\nKey Findings:")
print(f"  Isolation Forest ROC-AUC: {comparison.dict_if_metrics['ROC-AUC']:.4f}")
print(f"  LOF ROC-AUC: {comparison.dict_lof_metrics['ROC-AUC']:.4f}")
print(f"  Ensemble precision (both agree): {dict_ensemble_metrics['precision']:.4f}")
print(f"  Ensemble anomalies flagged: {dict_ensemble_metrics['flagged']:,}")